In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
if sys.platform != "darwin":
    os.environ["MUJOCO_GL"] = "egl"
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_DEFAULT_MATMUL_PRECISION"] = "highest"


import random
import time

import tqdm
import wandb
import numpy as np

import torch, gc
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
    SimpleReplayBufferGNN,
    save_params,
)

from fast_td3 import Critic
from fast_td3.actors import ActorEGNN, Actor, ActorEGNN2

In [3]:
from fast_td3.hyperparams import HumanoidBenchArgs

robot = "h1"

args = HumanoidBenchArgs(
    env_name=f"{robot}-stand-v0",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192,
    #actor_hidden_dim=384,
    actor_hidden_dim=512,
)

In [4]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

Using device: cuda:0


In [5]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = env.num_actions
n_obs = env.num_obs if type(env.num_obs) == int else env.num_obs[0]

In [6]:
checkpoint_path = "./models/egnn_h1-stand-v0_16envs_150001steps_c5dbf7_5000.pt"
obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
xanchor_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)

# Actor setup
actor = ActorEGNN(
    n_obs=n_obs,
    n_act=n_act,
    num_envs=args.num_envs,
    batch_size=args.batch_size,
    device=device,
    init_scale=args.init_scale,
    hidden_dim=64,
    n_layers=4,
    act_fn="relu",
    robot=robot,
    env_name="h1_run_v0",
)


torch_checkpoint = torch.load(
    f"{checkpoint_path}", map_location=device, weights_only=False
)
obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
xanchor_normalizer.load_state_dict(torch_checkpoint["xanchor_normalizer_state"])
actor.load_state_dict(torch_checkpoint["actor_state_dict"])

actor.eval()
xanchor_normalizer.eval()
obs_normalizer.eval()
normalize_obs = obs_normalizer.forward
normalize_xanchor = xanchor_normalizer.forward

In [7]:
# checkpoint_path = "./models/mlp_h1-stand-v0_16envs_50000steps_aac743_final.pt"
# obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
# xanchor_normalizer = EmpiricalNormalization(shape=(20, 3), device=device)
# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# actor = Actor(
#     n_obs=n_obs,
#     n_act=n_act,
#     num_envs=args.num_envs,
#     device=device,
#     init_scale=args.init_scale,
#     hidden_dim=args.actor_hidden_dim,
# )

# torch_checkpoint = torch.load(
#     f"{checkpoint_path}", map_location=device, weights_only=False
# )
# obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
# #xanchor_normalizer.load_state_dict(torch_checkpoint["xpos_normalizer_state"])
# actor.load_state_dict(torch_checkpoint["actor_state_dict"])

# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# print("actor parameters:", sum(p.numel() for p in actor.parameters()))

In [8]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
    """
    Convert a list of numpy arrays to an HTML5 video element.

    Args:
        frames (list): List of numpy arrays representing video frames
        fps (int): Frames per second for the video

    Returns:
        HTML object containing the video element
    """
    # Create a temporary file to store the video
    with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
        temp_filename = temp_file.name

    # Save frames as video
    imageio.mimsave(temp_filename, frames, fps=fps)

    # Read the video file and encode it to base64
    with open(temp_filename, "rb") as f:
        video_data = f.read()
    video_b64 = base64.b64encode(video_data).decode("utf-8")

    # Create HTML video element
    video_html = f"""
    <video width="640" height="480" controls>
        <source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
        Your browser does not support the video tag.
    </video>
    """

    # Clean up the temporary file
    os.unlink(temp_filename)

    return HTML(video_html)


def render_with_rollout():
    obs_normalizer.eval()
    xanchor_normalizer.eval()

    # Quick rollout for rendering
    if env_type == "humanoid_bench":
        obs, xanchor = env.reset()
        renders = [env.render()]

    for i in range(env.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):
            obs = normalize_obs(obs)
            xanchor = normalize_xanchor(xanchor)
            actions = actor(obs, xanchor)
        next_obs, _, done, _, next_xanchor = env.step(actions.float())
        if i % 2 == 0:
            renders.append(env.render())
        if done.any():
            break
        obs = next_obs
        xanchor = next_xanchor
        
    obs_normalizer.train()
    xanchor_normalizer.train()
    video_html = frames_to_video_html(renders, fps=30)
    display(video_html)


In [9]:

episode_returns = torch.zeros(env.num_envs, device=device)
episode_lengths = torch.zeros(env.num_envs, device=device)
done_masks = torch.zeros(env.num_envs, dtype=torch.bool, device=device)

obs, xanchor = env.reset()
for _ in range(env.max_episode_steps):
	with torch.no_grad(), autocast(
		device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
	):  
		# obs = normalize_obs(obs)
		# xanchor = normalize_xanchor(xanchor)
		actions = actor(obs, xanchor)

	next_obs, rewards, dones, _ , next_xanchor = env.step(actions.float())
	episode_returns = torch.where(
		~done_masks, episode_returns + rewards, episode_returns
	)
	episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
	done_masks = torch.logical_or(done_masks, dones)
	if done_masks.all():
		break
	obs = next_obs
	xanchor = next_xanchor

print(episode_returns.mean().item(), episode_lengths.mean().item())

30.104106903076172 60.0


In [10]:
# Quick rollout for rendering
if env_type == "humanoid_bench":
	obs, xanchor = env.reset()
	renders = [env.render()]

for i in range(env.max_episode_steps):
	with torch.no_grad(), autocast(
		device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
	):
		# if i == 1:
		# 	print("obs before normalized:", obs)
		# 	print("xanchor before normalized:", xanchor)
		# obs = normalize_obs(obs)
		# xanchor = normalize_xanchor(xanchor)
		# if i == 1:
		# 	print("obs after normalized:", obs)
		# 	print("xanchor after normalized:", xanchor)
		
		actions = actor(obs, xanchor)
	next_obs, _, done, _, next_xanchor = env.step(actions.float())
	if i % 2 == 0:
		renders.append(env.render())
	if done.any():
		break
	obs = next_obs
	xanchor = next_xanchor
	
obs_normalizer.train()
xanchor_normalizer.train()
video_html = frames_to_video_html(renders, fps=30)
display(video_html)